<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/MomentumBurst_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Momentum Burst Trading Strategy Rules

This strategy aims to identify short-term, explosive breakouts in strongly trending stocks.

### Core Rules:
1.  **Linear Trend:** Price must be above its 20-period Exponential Moving Average (EMA), and the 20 EMA must be rising.
2.  **Orderly Consolidation:** A tight, multi-day pullback/base (at least 3 days) dropping no more than 20% from recent 10-day highs.
3.  **No Big Breakdowns:** No massive high-volume selling days during consolidation. Daily ranges must stay quiet relative to the 20-day Average Daily Range (ADR).
4.  **Narrow Range Day:** The day *immediately before* the breakout must be a tight "inside" or quiet day (absolute move < 2%).
5.  **No 3-Days-In-A-Row:** Skip if the stock has closed green 3 consecutive days prior to today.
6.  **The 4% Breakout Trigger:** Today's candle must print a gain of >= 4%.
7.  **Close Near High:** Today's close must finish within the top 15% of the day's total high-low range.
8.  **Volume:** Today's volume must exceed yesterday's volume.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

In [2]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

In [3]:
import requests

def download_file_from_google_drive(url, destination):
    response = requests.get(url)
    if response.status_code == 200:
        with open(destination, 'wb') as f:
            f.write(response.content)
        print(f"File downloaded successfully to {destination}")
    else:
        print(f"Failed to download file from {url}. Status code: {response.status_code}")

option_volume_path = 'option_volume.csv'
download_file_from_google_drive(OptionVolume, option_volume_path)

File downloaded successfully to option_volume.csv


In [4]:
# Load the OptionVolume data and extract tickers
try:
    option_volume_df = pd.read_csv(option_volume_path)
    # Assuming the tickers are in a column named 'Ticker' or 'Symbol'
    # Adjust 'Ticker' if your CSV uses a different column name for symbols
    if 'Ticker' in option_volume_df.columns:
        option_volume_tickers = option_volume_df['Ticker'].tolist()
        print(f"Loaded {len(option_volume_tickers)} tickers from OptionVolume.")
    elif 'Symbol' in option_volume_df.columns:
        option_volume_tickers = option_volume_df['Symbol'].tolist()
        print(f"Loaded {len(option_volume_tickers)} tickers from OptionVolume.")
    else:
        print("Could not find 'Ticker' or 'Symbol' column in option_volume.csv")
        option_volume_tickers = [] # Fallback to empty list
except FileNotFoundError:
    print(f"Error: {option_volume_path} not found. Please ensure the file is downloaded correctly.")
    option_volume_tickers = []
except Exception as e:
    print(f"An error occurred while reading the OptionVolume file: {e}")
    option_volume_tickers = []

Loaded 100 tickers from OptionVolume.


In [5]:
def run_momentum_burst_backtester(tickers, start_date, end_date):
    results = []

    print(f"Downloading data for {len(tickers)} tickers...")
    # Fetch historical daily data for all tickers
    data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')

    # Handle single ticker edge case vs multi-ticker dataframes
    if len(tickers) == 1:
        ticker_data_dict = {tickers[0]: data}
    else:
        ticker_data_dict = {t: data[t].dropna() for t in tickers if t in data.columns.levels[0]}

    for ticker, df in ticker_data_dict.items():
        if df.empty or len(df) < 20: # Ensure enough data for indicators
            continue

        # --- SECTION 1: CALCULATE THE INDICATORS & CHECK STRATEGY RULES ---

        # 1. Price Changes & ADR (Average Daily Range over 20 periods)
        df['Daily_Range_Pct'] = (df['High'] - df['Low']) / df['Close'].shift(1)
        df['ADR'] = df['Daily_Range_Pct'].rolling(window=20).mean()
        df['Pct_Change'] = df['Close'].pct_change()

        # 2. Linear Trend Check (Simple approach: Price above 20 EMA, and 20 EMA rising)
        df['EMA20'] = df['Close'].ewm(span=20, adjust=False).mean()
        df['Linear_Trend'] = (df['Close'] > df['EMA20']) & (df['EMA20'] > df['EMA20'].shift(1))

        # 3. Consolidation Checks (Looking back at past 3 days before today)
        # Pullback max 20% from a rolling 10-day high
        df['Roll_High'] = df['High'].rolling(window=10).max()
        df['Pullback_Pct'] = (df['Roll_High'] - df['Close']) / df['Roll_High']
        df['Orderly_Pullback'] = df['Pullback_Pct'] <= 0.20

        # No big high-volume breakdowns (Price change below ADR on down days during pullback)
        df['Orderly_Daily_Range'] = df['Daily_Range_Pct'] <= (df['ADR'] * 1.5)

        # 4. Narrow range day check (Prior day's absolute movement < 2%)
        df['Prior_Day_Narrow'] = df['Pct_Change'].shift(1).abs() <= 0.02

        # 5. Up 3-days-in-a-row check
        df['Is_Up_Day'] = df['Pct_Change'] > 0
        df['Up_3_In_Row'] = (df['Is_Up_Day'].shift(1) & df['Is_Up_Day'].shift(2) & df['Is_Up_Day'].shift(3))

        # Loop through data to find triggers (leaving room at the end for Section 3 analysis)
        for i in range(5, len(df) - 5):
            # Evaluate Section 1 conditions for "Today" (index i)

            # A. Breakout Day Rules
            c_breakout_move = df['Pct_Change'].iloc[i] >= 0.04 # 4% Move

            # Close near daily high (within top 15% of the daily candle range)
            candle_range = df['High'].iloc[i] - df['Low'].iloc[i]
            c_close_near_high = True if candle_range == 0 else (df['High'].iloc[i] - df['Close'].iloc[i]) / candle_range <= 0.15

            # Volume higher than previous day
            c_volume_expansion = df['Volume'].iloc[i] > df['Volume'].iloc[i-1]

            # B. Backdrop Rules (Looking backward from yesterday/index i-1)
            c_linear_trend = df['Linear_Trend'].iloc[i-1]
            c_narrow_prior = df['Prior_Day_Narrow'].iloc[i]
            c_not_extended = not df['Up_3_In_Row'].iloc[i]

            # Consolidation over last 3 days
            c_orderly_base = (df['Orderly_Pullback'].iloc[i-3:i].all() and
                              df['Orderly_Daily_Range'].iloc[i-3:i].all())

            # TRIGGER VALIDATION
            if (c_breakout_move and c_close_near_high and c_volume_expansion and
                c_linear_trend and c_narrow_prior and c_not_extended and c_orderly_base):

                trigger_date = df.index[i]
                entry_price = df['Close'].iloc[i]

                # --- SECTION 3: ANALYZE FORWARD RETURNS (HOLDING 1 TO 5 DAYS) ---
                returns_dict = {
                    'Ticker': ticker,
                    'Trigger Date': trigger_date.strftime('%Y-%m-%d'),
                    'Entry Price': round(entry_price, 2)
                }

                for hold_days in range(1, 6):
                    future_close = df['Close'].iloc[i + hold_days]
                    forward_return = (future_close - entry_price) / entry_price
                    returns_dict[f'Day {hold_days} Return (%)'] = round(forward_return * 100, 2)

                results.append(returns_dict)

    return pd.DataFrame(results)



In [6]:
# --- EXECUTION BLOCK (Updated for OptionVolume tickers) ---
if __name__ == "__main__":
    # Define your manual list of tickers (Section 2)
    my_watch_list = option_volume_tickers # Using tickers from OptionVolume

    if not my_watch_list:
        print("No tickers available from OptionVolume. Please check the file and column name.")
    else:
        # Define backtest historical window
        start = "2024-01-01"
        end = "2026-06-01"

        df_backtest = run_momentum_burst_backtester(my_watch_list, start, end)

        if not df_backtest.empty:
            print("\n=== MOMENTUM BURST TRADES FOUND ===")
            print(df_backtest.to_string(index=False))

            print("\n=== STRATEGY PERFORMANCE SUMMARY ===")
            mean_returns = df_backtest[[f'Day {x} Return (%)' for x in range(1, 6)]].mean()
            print("Average Forward Return by Holding Period:")
            print(mean_returns)
        else:
            print("\nNo setups matched all constraints within this timeframe.")

/tmp/ipykernel_1526/1303565083.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')
[*********************100%***********************]  100 of 100 completed



=== MOMENTUM BURST TRADES FOUND ===
Ticker Trigger Date  Entry Price  Day 1 Return (%)  Day 2 Return (%)  Day 3 Return (%)  Day 4 Return (%)  Day 5 Return (%)
  TSLA   2024-05-21       186.60             -3.48             -6.89             -3.94             -5.28             -5.58
  TSLA   2024-09-19       243.92             -2.32              2.49              4.24              5.37              4.22
  TSLA   2024-12-13       436.23              6.14             10.00              0.89             -0.01             -3.48
  TSLA   2025-05-27       362.89             -1.65             -1.23             -4.53             -5.57             -5.13
  TSLA   2025-09-11       368.81              7.36             11.18             14.32             15.47             13.03
  TSLA   2025-12-03       446.74              1.74              1.85             -1.60             -0.35              1.05
   QQQ   2025-05-12       505.40              1.52              2.13              2.24              2.